# 📊 Bước 2: Exploratory Data Analysis (EDA) & Phân tích 5V
**Dự án:** Corporación Favorita Grocery Sales Forecasting  
**Thành viên:** Thành — Data & Infrastructure Engineer  
**Mục tiêu:** Khởi tạo Spark Session kết nối cụm, kiểm tra schema, phân tích đặc trưng 5V và xác định phân bố cửa hàng chuẩn bị cho chọn Subset đại diện.

### 1. Khởi tạo Spark Session kết nối Cụm Master

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd

# Khởi tạo SparkSession kết nối tới container spark-master
spark = SparkSession.builder \
    .appName("EDA_CorporacionFavorita") \
    .master("spark://spark-master:7077") \
    .config("spark.driver.host", "jupyter-pyspark") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.executor.memory", "2g") \
    .config("spark.executor.cores", "2") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print("✅ Spark Session Connected successfully!")
print(f"Spark version: {spark.version}")
print(f"Master: {spark.sparkContext.master}")

✅ Spark Session Connected successfully!
Spark version: 3.5.0
Master: spark://spark-master:7077


### 2. Đọc Schema các bảng dữ liệu

In [2]:
# Đọc các bảng phụ trợ (kích thước nhỏ)
stores = spark.read.csv("/home/jovyan/data/raw/stores.csv", header=True, inferSchema=True)
items = spark.read.csv("/home/jovyan/data/raw/items.csv", header=True, inferSchema=True)
oil = spark.read.csv("/home/jovyan/data/raw/oil.csv", header=True, inferSchema=True)
holidays = spark.read.csv("/home/jovyan/data/raw/holidays_events.csv", header=True, inferSchema=True)
transactions = spark.read.csv("/home/jovyan/data/raw/transactions.csv", header=True, inferSchema=True)

print("--- STORES SCHEMA ---")
stores.printSchema()

print("--- ITEMS SCHEMA ---")
items.printSchema()

print("--- OIL SCHEMA ---")
oil.printSchema()

--- STORES SCHEMA ---
root
 |-- store_nbr: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- type: string (nullable = true)
 |-- cluster: integer (nullable = true)

--- ITEMS SCHEMA ---
root
 |-- item_nbr: integer (nullable = true)
 |-- family: string (nullable = true)
 |-- class: integer (nullable = true)
 |-- perishable: integer (nullable = true)

--- OIL SCHEMA ---
root
 |-- date: date (nullable = true)
 |-- dcoilwtico: double (nullable = true)



### 3. Đọc Bảng Train (Dữ liệu Lớn ~125M dòng)

In [3]:
# Đọc train.csv (định dạng lazy evaluation của Spark sẽ không nạp hết vào RAM ngay)
train = spark.read.csv("/home/jovyan/data/raw/train.csv", header=True, inferSchema=True)
train.printSchema()
train.show(5)

root
 |-- id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- store_nbr: integer (nullable = true)
 |-- item_nbr: integer (nullable = true)
 |-- unit_sales: double (nullable = true)
 |-- onpromotion: boolean (nullable = true)

+---+----------+---------+--------+----------+-----------+
| id|      date|store_nbr|item_nbr|unit_sales|onpromotion|
+---+----------+---------+--------+----------+-----------+
|  0|2013-01-01|       25|  103665|       7.0|       NULL|
|  1|2013-01-01|       25|  105574|       1.0|       NULL|
|  2|2013-01-01|       25|  105575|       2.0|       NULL|
|  3|2013-01-01|       25|  108079|       1.0|       NULL|
|  4|2013-01-01|       25|  108701|       1.0|       NULL|
+---+----------+---------+--------+----------+-----------+
only showing top 5 rows



### 4. Phân tích 5V của Big Data (Dùng cho Báo cáo & Slide)

In [4]:
print("=" * 60)
print("1. VOLUME ANALYSIS")
print("=" * 60)
train_rows = train.count()
print(f"Train rows:        {train_rows:,}")
print(f"Stores rows:       {stores.count():,}")
print(f"Items rows:        {items.count():,}")
print(f"Transactions rows: {transactions.count():,}")
print(f"Oil rows:          {oil.count():,}")
print(f"Holidays rows:     {holidays.count():,}")

print("\n" + "=" * 60)
print("2. VARIETY ANALYSIS")
print("=" * 60)
print(f"Số nhóm hàng (family): {items.select('family').distinct().count()}")
print(f"Số thành phố:          {stores.select('city').distinct().count()}")
print(f"Số phân hạng cửa hàng: {stores.select('type').distinct().count()} types (A, B, C, D, E)")
print(f"Số cluster cửa hàng:   {stores.select('cluster').distinct().count()} clusters")

print("\n" + "=" * 60)
print("3. VELOCITY ANALYSIS")
print("=" * 60)
date_stats = train.agg(F.min("date").alias("min_date"), F.max("date").alias("max_date")).collect()[0]
print(f"Khoảng thời gian: Từ {date_stats['min_date']} đến {date_stats['max_date']}")
print("Tần suất thu thập: Daily Point-of-Sale (POS) granularity")

print("\n" + "=" * 60)
print("4. VERACITY ANALYSIS (Chất lượng & Dữ liệu nhiễu)")
print("=" * 60)
neg_sales = train.filter(F.col("unit_sales") < 0).count()
print(f"Số bản ghi trả lại (unit_sales < 0): {neg_sales:,} ({neg_sales/train_rows*100:.3f}%)")
null_onpromo = train.filter(F.col("onpromotion").isNull()).count()
print(f"Số bản ghi onpromotion bị null:     {null_onpromo:,} ({null_onpromo/train_rows*100:.3f}%)")
null_oil = oil.filter(F.col("dcoilwtico").isNull()).count()
print(f"Số ngày giá dầu bị trống (cuối tuần/nghỉ): {null_oil:,} ngày")

print("\n" + "=" * 60)
print("5. VALUE ANALYSIS")
print("=" * 60)
print("Dự báo chính xác nhu cầu tiêu thụ tại từng cửa hàng giúp Corporación Favorita:")
print("- Tối ưu quản lý tồn kho, giảm lãng phí thực phẩm tươi sống (perishable goods)")
print("- Ngăn chặn tình trạng đứt gãy nguồn cung (stockouts)")

1. VOLUME ANALYSIS
Train rows:        125,497,040
Stores rows:       54
Items rows:        4,100
Transactions rows: 83,488
Oil rows:          1,218
Holidays rows:     350

2. VARIETY ANALYSIS
Số nhóm hàng (family): 33
Số thành phố:          22
Số phân hạng cửa hàng: 5 types (A, B, C, D, E)
Số cluster cửa hàng:   17 clusters

3. VELOCITY ANALYSIS
Khoảng thời gian: Từ 2013-01-01 đến 2017-08-15
Tần suất thu thập: Daily Point-of-Sale (POS) granularity

4. VERACITY ANALYSIS (Chất lượng & Dữ liệu nhiễu)
Số bản ghi trả lại (unit_sales < 0): 7,795 (0.006%)
Số bản ghi onpromotion bị null:     21,657,651 (17.257%)
Số ngày giá dầu bị trống (cuối tuần/nghỉ): 43 ngày

5. VALUE ANALYSIS
Dự báo chính xác nhu cầu tiêu thụ tại từng cửa hàng giúp Corporación Favorita:
- Tối ưu quản lý tồn kho, giảm lãng phí thực phẩm tươi sống (perishable goods)
- Ngăn chặn tình trạng đứt gãy nguồn cung (stockouts)


### 5. Phân tích Phân bố Cửa hàng để Chọn 10 Cửa hàng Đại diện (Stratified Sampling)

In [5]:
# Phân bố theo type và cluster
print("--- Phân bố Stores theo Type ---")
stores.groupBy("type").count().orderBy("type").show()

print("--- Phân bố Stores theo Thành phố (Top 10) ---")
stores.groupBy("city").count().orderBy(F.desc("count")).show(10)

print("--- Chi tiết tất cả 54 cửa hàng ---")
stores.orderBy("type", "cluster", "city").show(54, truncate=False)

--- Phân bố Stores theo Type ---
+----+-----+
|type|count|
+----+-----+
|   A|    9|
|   B|    8|
|   C|   15|
|   D|   18|
|   E|    4|
+----+-----+

--- Phân bố Stores theo Thành phố (Top 10) ---
+-------------+-----+
|         city|count|
+-------------+-----+
|        Quito|   18|
|    Guayaquil|    8|
|Santo Domingo|    3|
|       Cuenca|    3|
|        Manta|    2|
|    Latacunga|    2|
|      Machala|    2|
|       Ambato|    2|
|     Riobamba|    1|
|     Guaranda|    1|
+-------------+-----+
only showing top 10 rows

--- Chi tiết tất cả 54 cửa hàng ---
+---------+-------------+------------------------------+----+-------+
|store_nbr|city         |state                         |type|cluster|
+---------+-------------+------------------------------+----+-------+
|44       |Quito        |Pichincha                     |A   |5      |
|52       |Manta        |Manabi                        |A   |11     |
|45       |Quito        |Pichincha                     |A   |11     |
|49       |Q